# RAG Databricks Bluetab - Multi-Model Serving Endpoint Creation

## Overview
This notebook creates a single serving endpoint to host both the LLM and the embedding models, enabling real-time inference for both functionalities from one location.

## Features
- Automated endpoint creation and updates with error handling.
- Serves multiple models (LLM and Embedding) from a single endpoint.
- Configurable workload sizes and scaling options for each model.
- Health checking and status monitoring.
- Integration with MLflow Unity Catalog model registry.
- Parameterized deployment for different environments.

## Endpoint Configuration
- **Workload Size**: Configurable per model (Small, Medium, Large).
- **Auto-scaling**: Enabled with scale-to-zero for cost optimization.
- **Monitoring**: Built-in logging and metrics.

## Dependencies
- Run the `00 Configuration and Utils` notebook first.
- Ensure both the LLM and the embedding models are registered in the MLflow Model Registry.

In [0]:
# Crear widgets de configuración
# Core Configuration
dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
dbutils.widgets.text("schema_name", "rag", "Schema Name")
dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")

# Endpoint Configuration
dbutils.widgets.text("endpoint_name", "rag_bluetab_endpoint", "Unified Endpoint Name")
dbutils.widgets.text("endpoint_suffix", "", "Endpoint Name Suffix (optional)")

# LLM Model Configuration
dbutils.widgets.text("llm_model_name", "flan_t5_base_model", "LLM Model Name")
dbutils.widgets.text("llm_model_version", "latest", "LLM Model Version (or 'latest')")
dbutils.widgets.dropdown("llm_workload_size", "Small", ["Small", "Medium", "Large"], "LLM Workload Size")
dbutils.widgets.dropdown("llm_scale_to_zero", "true", ["true", "false"], "LLM Enable Scale to Zero")

# Embedding Model Configuration
dbutils.widgets.text("embedding_model_name", "simple_embedding_model_bluetab", "Embedding Model Name")
dbutils.widgets.text("embedding_model_version", "latest", "Embedding Model Version (or 'latest')")
dbutils.widgets.dropdown("embedding_workload_size", "Small", ["Small", "Medium", "Large"], "Embedding Workload Size")
dbutils.widgets.dropdown("embedding_scale_to_zero", "true", ["true", "false"], "Embedding Enable Scale to Zero")

# MLflow Configuration
dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# MLflow run management
dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
# Obtener valores de los widgets
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
ENVIRONMENT = dbutils.widgets.get("environment")

# Endpoint configuration
ENDPOINT = dbutils.widgets.get("endpoint_name")
ENDPOINT_SUFFIX = dbutils.widgets.get("endpoint_suffix")

# LLM Config
LLM_MODEL_NAME = dbutils.widgets.get("llm_model_name")
LLM_MODEL_VERSION = dbutils.widgets.get("llm_model_version")
LLM_WORKLOAD_SIZE = dbutils.widgets.get("llm_workload_size")
LLM_SCALE_TO_ZERO = dbutils.widgets.get("llm_scale_to_zero").lower() == "true"
LLM_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{LLM_MODEL_NAME}"

# Embedding Config
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model_name")
EMBEDDING_MODEL_VERSION = dbutils.widgets.get("embedding_model_version")
EMBEDDING_WORKLOAD_SIZE = dbutils.widgets.get("embedding_workload_size")
EMBEDDING_SCALE_TO_ZERO = dbutils.widgets.get("embedding_scale_to_zero").lower() == "true"
EMBEDDING_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{EMBEDDING_MODEL_NAME}"

# MLflow configuration
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# Variables globales para gestión de parent/child runs
PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
CURRENT_RUN = dbutils.widgets.get("current_run") or None

print("¡Configuración cargada correctamente!")
print(f"Environment: {ENVIRONMENT}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Schema: {SCHEMA_NAME}")

In [0]:
from config.config_utils import (
    start_child_run, 
    end_child_run, 
    log_step,
    setup_mlflow_experiment
)

# Configurar MLflow experiment
setup_mlflow_experiment(EXPERIMENT_NAME)

In [0]:
# Iniciar child run para esta tarea
start_child_run("05_create_multi_model_endpoint", environment=ENVIRONMENT)

In [0]:
# Build endpoint name with optional suffix
if ENDPOINT_SUFFIX:
    ENDPOINT_NAME = f"{ENDPOINT}_{ENDPOINT_SUFFIX}"
else:
    ENDPOINT_NAME = ENDPOINT

print(f"Unified Endpoint Configuration:")
print(f"  Name: {ENDPOINT_NAME}")
print("\n--- LLM Model ---")
print(f"  Model: {LLM_MODEL_FULL}")
print(f"  Version: {LLM_MODEL_VERSION}")
print(f"  Workload Size: {LLM_WORKLOAD_SIZE}")
print(f"  Scale to Zero: {LLM_SCALE_TO_ZERO}")

print("--- Embedding Model ---")
print(f"  Model: {EMBEDDING_MODEL_FULL}")
print(f"  Version: {EMBEDDING_MODEL_VERSION}")
print(f"  Workload Size: {EMBEDDING_WORKLOAD_SIZE}")
print(f"  Scale to Zero: {EMBEDDING_SCALE_TO_ZERO}")

In [0]:
import mlflow
from mlflow.deployments import get_deploy_client

mlflow.log_param("step", "multi_model_endpoint_creation")
mlflow.log_param("environment", ENVIRONMENT)
mlflow.log_param("endpoint_name", ENDPOINT_NAME)
mlflow.log_param("llm_model_name", LLM_MODEL_FULL)
mlflow.log_param("embedding_model_name", EMBEDDING_MODEL_FULL)

log_step("endpoint_creation", "started", f"Creating/updating endpoint {ENDPOINT_NAME}")

# Initialize deployment client
client = get_deploy_client("databricks")

log_step("client_initialization", "success", "Deployment client initialized")

In [0]:
# Check if endpoint already exists
log_step("endpoint_check", "started", f"Checking if endpoint {ENDPOINT_NAME} exists")

try:
    existing_endpoints = client.list_endpoints()
    endpoint_names = [ep.get('name', '') for ep in existing_endpoints]
    
    endpoint_exists = ENDPOINT_NAME in endpoint_names
    
    if endpoint_exists:
        log_step("endpoint_check", "found", f"Endpoint {ENDPOINT_NAME} already exists. Will perform an update if needed.")
        print(f"⚠️  Endpoint '{ENDPOINT_NAME}' already exists! Will check for necessary updates.")
        mlflow.log_param("endpoint_exists", True)
    else:
        log_step("endpoint_check", "not_found", f"Endpoint {ENDPOINT_NAME} does not exist. Will create a new one.")
        print(f"✅ Endpoint '{ENDPOINT_NAME}' does not exist. A new one will be created.")
        mlflow.log_param("endpoint_exists", False)
        
except Exception as e:
    log_step("endpoint_check", "error", f"Error checking endpoints: {e}")
    endpoint_exists = False
    mlflow.log_param("endpoint_check_error", str(e))
    raise e

In [0]:
# Enhanced endpoint creation/update logic for multiple models
from mlflow.tracking import MlflowClient
import mlflow

mlflow.set_registry_uri("databricks-uc")
mlflow_client = MlflowClient()
deploy_client = get_deploy_client("databricks")

def resolve_model_version(model_full_name: str, requested_version: str) -> str:
    """Resolve the model version, handling 'latest' and validation."""
    if requested_version.lower() == "latest":
        model_versions = mlflow_client.search_model_versions(f"name='{model_full_name}'")
        if model_versions:
            latest_version = max([int(mv.version) for mv in model_versions])
            resolved_version = str(latest_version)
            print(f"Resolved 'latest' for model '{model_full_name}' to version {resolved_version}")
        else:
            raise ValueError(f"No versions found for model '{model_full_name}'. Cannot resolve 'latest'.")
    else:
        resolved_version = requested_version
        print(f"Using specified version {resolved_version} for model '{model_full_name}'")
    return resolved_version

# 1. Resolve versions for both models
resolved_llm_version = resolve_model_version(LLM_MODEL_FULL, LLM_MODEL_VERSION)
resolved_embedding_version = resolve_model_version(EMBEDDING_MODEL_FULL, EMBEDDING_MODEL_VERSION)

mlflow.log_param("resolved_llm_version", resolved_llm_version)
mlflow.log_param("resolved_embedding_version", resolved_embedding_version)

# Generar nombres de entidad
llm_entity_name = f"{LLM_MODEL_NAME.replace('_', '-')}-entity"
embedding_entity_name = f"{EMBEDDING_MODEL_NAME.replace('_', '-')}-entity"

# 2. Define the multi-model endpoint configuration
endpoint_config = {
    "served_entities": [
        {
            "name": llm_entity_name,
            "entity_name": LLM_MODEL_FULL,
            "entity_version": resolved_llm_version,
            "workload_size": LLM_WORKLOAD_SIZE,
            "scale_to_zero_enabled": LLM_SCALE_TO_ZERO
        },
        {
            "name": embedding_entity_name,
            "entity_name": EMBEDDING_MODEL_FULL,
            "entity_version": resolved_embedding_version,
            "workload_size": EMBEDDING_WORKLOAD_SIZE,
            "scale_to_zero_enabled": EMBEDDING_SCALE_TO_ZERO
        }
    ],
    # --- AÑADIDO: Bloque de configuración de tráfico OBLIGATORIO ---
    "traffic_config": {
        "routes": [
            {
                "served_model_name": llm_entity_name,
                "traffic_percentage": 50
            },
            {
                "served_model_name": embedding_entity_name,
                "traffic_percentage": 50
            }
        ]
    }
}

mlflow.log_dict(endpoint_config, "endpoint_config.json")
print("\nTarget Endpoint Configuration Generated:")
print(f"  - LLM Model: {LLM_MODEL_FULL} (v{resolved_llm_version}) with workload '{LLM_WORKLOAD_SIZE}'")
print(f"  - Embedding Model: {EMBEDDING_MODEL_FULL} (v{resolved_embedding_version}) with workload '{EMBEDDING_WORKLOAD_SIZE}'")

# 3. Create or Update the endpoint
if not endpoint_exists:
    # CREATE NEW ENDPOINT
    log_step("endpoint_creation", "started", f"Creating new multi-model endpoint {ENDPOINT_NAME}")
    try:
        print(f"\nCreating new endpoint '{ENDPOINT_NAME}'...")
        endpoint = deploy_client.create_endpoint(name=ENDPOINT_NAME, config=endpoint_config)
        mlflow.log_param("endpoint_action", "created")
        log_step("endpoint_creation", "success", f"Endpoint {ENDPOINT_NAME} created successfully")
        print(f"✅ Endpoint '{ENDPOINT_NAME}' creation initiated successfully!")
    except Exception as e:
        log_step("endpoint_creation", "failed", f"Error creating endpoint: {e}")
        print(f"❌ Error creating endpoint: {e}")
        raise e
else:
    # UPDATE EXISTING ENDPOINT
    log_step("endpoint_update", "started", f"Updating existing endpoint {ENDPOINT_NAME}")
    try:
        print(f"\nUpdating endpoint '{ENDPOINT_NAME}' with new configuration...")
        updated_endpoint = deploy_client.update_endpoint(endpoint=ENDPOINT_NAME, config=endpoint_config)
        mlflow.log_param("endpoint_action", "updated")
        log_step("endpoint_update", "success", f"Endpoint {ENDPOINT_NAME} updated successfully")
        print(f"✅ Endpoint '{ENDPOINT_NAME}' update initiated successfully!")
    except Exception as e:
        log_step("endpoint_update", "failed", f"Error updating endpoint: {e}")
        print(f"❌ Error updating endpoint: {e}")
        raise e

# 4. Display summary
try:
    workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
    endpoint_url = f"https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"
    mlflow.log_param("endpoint_url", endpoint_url)
    print(f"\nEndpoint will be available at: {endpoint_url}")
except Exception as e:
    print("Could not automatically construct endpoint URL.")

In [0]:
# Wait for endpoint to be ready and monitor status
import time

log_step("endpoint_monitoring", "started", "Monitoring endpoint deployment status")

max_wait_time = 1800  # 30 minutes max wait
check_interval = 30   # Check every 30 seconds
elapsed_time = 0

print(f"⏳ Waiting for endpoint to be ready (max {max_wait_time//60} minutes)...")

while elapsed_time < max_wait_time:
    try:
        endpoint_status = deploy_client.get_endpoint(ENDPOINT_NAME)
        state = endpoint_status.get('state', {})
        ready_status = state.get('ready', 'UNKNOWN')
        config_update = state.get('config_update', 'UNKNOWN')
        
        print(f"⏱️  Time: {elapsed_time//60}m {elapsed_time%60}s - Status: {ready_status} - Config: {config_update}")
        
        if ready_status == 'READY':
            log_step("endpoint_monitoring", "success", f"Endpoint is ready after {elapsed_time} seconds")
            mlflow.log_metric("deployment_time_seconds", elapsed_time)
            print(f"\n✅ Endpoint '{ENDPOINT_NAME}' is now READY!")
            break
        elif ready_status == 'FAILED':
            log_step("endpoint_monitoring", "failed", "Endpoint deployment failed")
            print(f"\n❌ Endpoint deployment failed! Check the Serving UI for details.")
            break
            
    except Exception as e:
        print(f"⚠️  Error checking status: {e}")
        
    time.sleep(check_interval)
    elapsed_time += check_interval
else:
    log_step("endpoint_monitoring", "timeout", f"Timeout after {max_wait_time} seconds")
    print(f"\n⏰ Timeout: Endpoint not ready after {max_wait_time//60} minutes.")

In [0]:
# Test both served models in the endpoint
import requests
import json

def test_endpoint_model(model_entity_name: str, test_payload: dict):
    log_step(f"endpoint_testing_{model_entity_name}", "started", f"Testing model entity {model_entity_name}")
    print(f"\n🧪 Testing model entity: {model_entity_name}...")
    
    try:
        # Get workspace URL and token for API request
        workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
        # NOTE: This token is scoped to the current user and job. For production, use a service principal.
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        
        headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
        url = f'https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations/{model_entity_name}'

        response = requests.post(url, headers=headers, data=json.dumps(test_payload))
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        
        response_json = response.json()
        print(f"✅ Test successful for {model_entity_name}!")
        print(f"   Response sample: {str(response_json)[:200]}...")
        log_step(f"endpoint_testing_{model_entity_name}", "success", f"Test for {model_entity_name} was successful.")
        mlflow.log_param(f"test_status_{model_entity_name}", "success")
        
    except Exception as e:
        error_message = f"Endpoint test failed for {model_entity_name}: {e}"
        print(f"❌ {error_message}")
        log_step(f"endpoint_testing_{model_entity_name}", "failed", error_message)
        mlflow.log_param(f"test_status_{model_entity_name}", "failed")
        mlflow.log_param(f"test_error_{model_entity_name}", str(e))

# --- Test Payload for LLM Model ---
llm_entity_name = f"{LLM_MODEL_NAME.replace('_', '-')}-entity"
llm_test_data = {
    "dataframe_split": {
        "columns": ["messages"],
        "data": [[[
            {"role": "user", "content": "Contexto: La paella valenciana es un plato de arroz. Pregunta: ¿De dónde es la paella? Respuesta:"}
        ]]]
    }
}
test_endpoint_model(llm_entity_name, llm_test_data)

# --- Test Payload for Embedding Model ---
embedding_entity_name = f"{EMBEDDING_MODEL_NAME.replace('_', '-')}-entity"
embedding_test_data = {
    "dataframe_split": {
        "columns": ["input"],
        "data": [["This is a test sentence for the embedding model"]]
    }
}
test_endpoint_model(embedding_entity_name, embedding_test_data)

In [0]:
# Provide final summary
from config.config_utils import build_endpoint_url

log_step("endpoint_creation", "completed", "Endpoint creation/update process finished")

print("="*60)
print("MULTI-MODEL ENDPOINT CREATION SUMMARY")
print("="*60)
print(f"Endpoint Name: {ENDPOINT_NAME}")
print(f"Environment: {ENVIRONMENT}")

print(f"Served LLM: {LLM_MODEL_FULL} (v{resolved_llm_version})")
print(f"Served Embedding Model: {EMBEDDING_MODEL_FULL} (v{resolved_embedding_version})")
print()

try:
    final_status = deploy_client.get_endpoint(ENDPOINT_NAME)
    ready_state = final_status.get('state', {}).get('ready', 'UNKNOWN')
    print(f"Final Status: {ready_state}")
    
    if ready_state == 'READY':
        print("✅ Endpoint is ready for use!")
        endpoint_url = build_endpoint_url(ENDPOINT_NAME)
        print(f"\n🔗 Base Endpoint URL: {endpoint_url}")
        print(f"   - Invoke LLM: /invocations/{llm_entity_name}")
        print(f"   - Invoke Embedding: /invocations/{embedding_entity_name}")
    else:
        print(f"⚠️  Endpoint status is not READY. Check the Databricks Serving UI for more details.")
        
except Exception as e:
    print(f"❌ Could not get final status: {e}")

In [0]:
# Finalizar child run
try:
    end_child_run("success")
    print("\n✅ Child run finalizada correctamente.")
except Exception as e:
    print(f"⚠️ Error finalizando child run: {e}")
    end_child_run("failed")